In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Load CIFAR-10 dataset
cifar_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.ToTensor()  # Converts to tensor and normalizes to [0, 1]
)

print(f"CIFAR-10 loaded: {len(cifar_data)} images")
print(f"Image shape: {cifar_data[0][0].shape}")  # Should be (3, 32, 32)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Define transformations for the training and test sets
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Load CIFAR-10 datasets
trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Create DataLoaders
trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(f"Number of training samples: {len(trainset)}")
print(f"Number of test samples: {len(testset)}")

In [ ]:
def imshow(img):
    img = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Show images
imshow(torchvision.utils.make_grid(images[:4]))
# Print labels
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))

In [ ]:
class VGG16_CIFAR10(nn.Module):
    def __init__(self, num_classes=10):
        super(VGG16_CIFAR10, self).__init__()
        # Load pre-trained VGG16 model
        vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

        # Freeze all parameters in the feature extractor
        for param in vgg16.features.parameters():
            param.requires_grad = False

        self.features = vgg16.features

        # Replace the classifier with a new one for CIFAR-10
        # VGG16's original classifier expects input features from avgpool of size 512*7*7 (for 224x224 input)
        # For 32x32 input, after features, the size will be smaller. Let's adapt.
        # Example: For 32x32 input, after VGG features, output size is usually 512 * 1 * 1 for VGG-16.
        # We need to flatten this to 512.

        # A common practice is to calculate the output size of features layer with a dummy tensor
        # dummy_input = torch.randn(1, 3, 32, 32)
        # output_features = self.features(dummy_input)
        # num_features = output_features.view(output_features.size(0), -1).size(1)
        # print(f"Number of features from VGG16 features for 32x32 input: {num_features}")
        # For VGG16 on 32x32, it's typically 512 (after features and AdaptiveAvgPool2d)

        self.classifier = nn.Sequential(
            nn.Linear(512 * 1 * 1, 4096), # Assuming features output 512x1x1 for 32x32 input
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        )

        # To handle potential variations in feature map size, we can use AdaptiveAvgPool2d
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1) # Flatten all dimensions except batch
        x = self.classifier(x)
        return x

model = VGG16_CIFAR10(num_classes=len(classes)).to(device)
print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.classifier.parameters(), lr=0.001, momentum=0.9)

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # Zero the parameter gradients

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

def validate_epoch(model, dataloader, criterion, device):
    model.eval() # Set the model to evaluation mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad(): # Disable gradient calculation
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

In [ ]:
class ColorizationDataset(Dataset):
    """
    A dataset for image colorization.
    Returns (grayscale_image, color_image) pairs.

    Args:
        cifar_dataset: The CIFAR-10 dataset (already transformed to tensors)
    """

    def __init__(self, cifar_dataset):
        # TO DO: Store the dataset
        pass

    def __len__(self):
        # TO DO: Return the number of samples
        pass

    def rgb_to_grayscale(self, img):
        """
        Convert an RGB image to grayscale.

        Args:
            img: Tensor of shape (3, H, W) with values in [0, 1]

        Returns:
            Tensor of shape (1, H, W) with values in [0, 1]
        """
        # TO DO: Implement RGB to grayscale conversion
        # Hint: Gray = 0.299 * R + 0.587 * G + 0.114 * B
        pass

    def __getitem__(self, idx):
        """
        Returns:
            grayscale_image: Tensor of shape (1, H, W)
            color_image: Tensor of shape (3, H, W)
        """
        # TO DO: Get the color image and convert to grayscale
        # Return (grayscale_image, color_image)
        pass

In [ ]:
# Test your implementation
colorization_dataset = ColorizationDataset(cifar_data)

# Get a sample
gray_img, color_img = colorization_dataset[0]

print(f"Grayscale image shape: {gray_img.shape}")  # Should be (1, 32, 32)
print(f"Color image shape: {color_img.shape}")      # Should be (3, 32, 32)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(gray_img.squeeze(), cmap='gray')
axes[0].set_title('Grayscale (Input)')
axes[0].axis('off')
axes[1].imshow(color_img.permute(1, 2, 0))
axes[1].set_title('Color (Target)')
axes[1].axis('off')
plt.show()

In [ ]:
# Test with DataLoader
dataloader = DataLoader(colorization_dataset, batch_size=8, shuffle=True)

gray_batch, color_batch = next(iter(dataloader))
print(f"Batch grayscale shape: {gray_batch.shape}")  # Should be (8, 1, 32, 32)
print(f"Batch color shape: {color_batch.shape}")      # Should be (8, 3, 32, 32)